# Instituto Federal Goiano (IF Goiano)## Pós-Graduação em Inteligência Artificial Aplicada### Trabalho de Conclusão de Trilha de Aprendizagem: Trilha I**Título:** AcolheMente Escolar PB — Motor de Inferência Lógico para Acolhimento em Saúde Mental Escolar**Aluno:** Leonardo Maximino Bernardo**Orientação:** Trilha I — Lógica Proposicional, Sistemas Baseados em Conhecimento e Inferência por Resolução**Versão:** v0.3.0 (7 variáveis, 6 regras — motor congelado por ADR-011)---

### 1. Apresentação do Aluno e ContextoEste trabalho foi desenvolvido como atividade final da **Trilha I** do curso de Pós-Graduação em Inteligência Artificial Aplicada do IF Goiano.O projeto **AcolheMente Escolar PB** implementa um Sistema Baseado em Conhecimento (SBC) utilizando **Lógica Proposicional com Inferência por Resolução** para identificar cenários de acolhimento prioritário em saúde mental escolar, com foco no contexto da Paraíba.**Inspiração pedagógica:** Este notebook se inspira no notebook-base `Resposta_Trabalho_Trilha_I.ipynb` fornecido como referência da Trilha I, expandindo cautelosamente o modelo de 5 para 7 variáveis proposicionais para incorporar contexto comportamental e capacidade institucional.

### 2. Introdução e Definição do ProblemaA **Pesquisa Nacional de Saúde do Escolar (PeNSE 2024)** revelou indicadores preocupantes de saúde mental entre adolescentes brasileiros, particularmente na região Nordeste e no estado da Paraíba.**Problema:** Como apoiar equipes pedagógicas na identificação de cenários que demandam acolhimento prioritário em saúde mental, sem criar sistemas de diagnóstico automatizado, ranking de estudantes ou vigilância discente?**Solução proposta:** Um motor lógico determinístico baseado em regras explícitas que:- Opera sobre **7 variáveis proposicionais** extraídas da PeNSE 2024;- Utiliza **inferência por resolução** sobre base de conhecimento em CNF;- Gera apenas uma saída operacional: **Acolhimento Humano Prioritário (A)**;- Exige **revisão humana obrigatória** antes de qualquer ação;- Respeita integralmente LGPD, ECA e PSE.> **Nota ética:** Este sistema NÃO diagnostica, NÃO prediz e NÃO classifica indivíduos. Toda saída é sugestão para revisão por profissional habilitado.

### 3. Metodologia e Escolha da Técnica de IA#### 3.1 Lógica ProposicionalA técnica central é a **Lógica Proposicional** aplicada a um Sistema Baseado em Conhecimento (SBC). As variáveis são binárias (verdadeiro/falso), derivadas de questões da PeNSE 2024, e as regras são definidas explicitamente por especialistas — não aprendidas por máquina.#### 3.2 Inferência por ResoluçãoO método de inferência utilizado é a **Resolução por Refutação**:1. Converter todas as regras para **Forma Normal Conjuntiva (CNF)**;2. Adicionar a negação da query (¬A) à base de conhecimento;3. Aplicar resolução binária repetidamente;4. Se a cláusula vazia (⊥) for derivada → A é consequência lógica;5. Se a saturação ocorrer sem ⊥ → A não é inferível.#### 3.3 Justificativa de não usar diagnóstico automatizadoO sistema opera no nível de **triagem institucional**, não de diagnóstico clínico. Nenhuma variável representa condição clínica (depressão, TDAH, autismo, TOC). Todas as variáveis descrevem **situações observáveis e autorreferidas** na PeNSE 2024.#### 3.4 Por que avançar de 5 para 7 variáveis?> A versão-base com 5 variáveis preservava máxima parcimônia e alinhamento direto com o notebook exemplar da Trilha I. A evolução para 7 variáveis foi adotada de forma **cautelosa** para incorporar dois elementos essenciais no contexto escolar: **contexto comportamental** e **capacidade institucional de resposta**. As variáveis C e I não ampliam o sistema para diagnóstico ou predição clínica; elas qualificam o ambiente de decisão e fortalecem a utilidade institucional do motor, mantendo explicabilidade e revisão humana obrigatória.

### 4. Implementação da Solução#### 4.1 Configuração do Ambiente

In [1]:
# Importações necessáriasimport sys, ossys.path.insert(0, os.path.join(os.getcwd(), '..'))# Módulos do AcolheMentefrom src.acolhemente.rule_graph import (    PROPOSITIONAL_VARIABLES, NUCLEAR_VARIABLES, CONTEXTUAL_VARIABLES,    ALL_RULES, ALL_CNFS,    E, B, V, S, A, C, I,    R1, R2, R3, R4, R5, R6,    R1_CNF, R2_CNF, R3_CNF, R4_CNF, R5_CNF, R6_CNF,    build_rule_graph, build_governance_graph, export_graph_png,)from src.acolhemente.graph_schema import EntityTypeprint("✅ Módulos AcolheMente carregados com sucesso.")print(f"   Variáveis proposicionais: {len(PROPOSITIONAL_VARIABLES)}")print(f"   Nucleares: {len(NUCLEAR_VARIABLES)}")print(f"   Contextuais: {len(CONTEXTUAL_VARIABLES)}")print(f"   Regras lógicas: {len(ALL_RULES)}")

#### 4.2 Tabela das 7 Variáveis Proposicionais

In [2]:
# Tabela das 7 variáveis proposicionaisprint("=" * 90)print(f"{'Var':<4} {'Tipo':<12} {'Semântica':<45} {'Fontes'}")print("=" * 90)for var in PROPOSITIONAL_VARIABLES:    tipo = "Nuclear" if var in NUCLEAR_VARIABLES else "Contextual"    # Extrair fontes da descrição    desc = var.description    print(f"{var.name:<4} {tipo:<12} {desc}")print("=" * 90)print(f"\nTotal: {len(PROPOSITIONAL_VARIABLES)} variáveis "      f"({len(NUCLEAR_VARIABLES)} nucleares + {len(CONTEXTUAL_VARIABLES)} contextuais)")

#### 4.3 Base de Conhecimento: Regras R1–R6

In [3]:
# Base de conhecimento: 6 regras lógicasprint("\n" + "=" * 70)print("BASE DE CONHECIMENTO — Regras Lógicas")print("=" * 70)for rule, cnf in zip(ALL_RULES, ALL_CNFS):    print(f"  {rule.name}: {rule.description:<20}  │  CNF: {cnf.description}")print("=" * 70)print(f"\nTotal: {len(ALL_RULES)} regras, {len(ALL_CNFS)} cláusulas CNF")print("\n⚠️  Nota: C e I são variáveis contextuais de modulação.")print("   C nunca infere A isoladamente (R5 exige E ∧ B ∧ C).")print("   I nunca infere A isoladamente (R6 exige V ∧ I).")

#### 4.4 Motor de Inferência por Resolução

In [4]:
# =====================================================================# MOTOR DE INFERÊNCIA POR RESOLUÇÃO# =====================================================================def cnf_knowledge_base():    """Retorna a base de conhecimento em CNF como lista de conjuntos."""    return [        {"~S", "A"},           # R1: ¬S ∨ A        {"~V", "~E", "A"},     # R2: ¬V ∨ ¬E ∨ A        {"~E", "~B", "A"},     # R3: ¬E ∨ ¬B ∨ A        {"~V", "~B", "A"},     # R4: ¬V ∨ ¬B ∨ A        {"~E", "~B", "~C", "A"},  # R5: ¬E ∨ ¬B ∨ ¬C ∨ A        {"~V", "~I", "A"},     # R6: ¬V ∨ ¬I ∨ A    ]def negate(literal):    """Nega um literal: A → ~A, ~A → A."""    return literal[1:] if literal.startswith("~") else f"~{literal}"def resolve(clause1, clause2):    """Aplica resolução binária entre duas cláusulas."""    resolvents = []    for lit in clause1:        neg = negate(lit)        if neg in clause2:            new_clause = (clause1 - {lit}) | (clause2 - {neg})            resolvents.append(frozenset(new_clause))    return resolventsdef resolution_proof(kb, query_var="A"):    """    Prova por resolução: tenta derivar A a partir da KB + fatos.        Adiciona ¬A à KB e tenta derivar a cláusula vazia (⊥).    Se ⊥ for derivada → A é consequência lógica.    """    clauses = [frozenset(c) for c in kb]    clauses.append(frozenset({negate(query_var)}))  # Adiciona ¬A        steps = []    new = set()        while True:        n = len(clauses)        pairs = [(clauses[i], clauses[j])                  for i in range(n) for j in range(i+1, n)]                for ci, cj in pairs:            resolvents = resolve(set(ci), set(cj))            for r in resolvents:                if len(r) == 0:  # Cláusula vazia → contradição!                    steps.append(f"  Resolveu {set(ci)} × {set(cj)} → ⊥ (CONTRADIÇÃO)")                    return True, steps                if r not in new and r not in set(map(frozenset, [set(c) for c in clauses])):                    new.add(r)                    steps.append(f"  Resolveu {set(ci)} × {set(cj)} → {set(r)}")                if new.issubset(set(map(frozenset, [set(c) for c in clauses]))):            return False, steps  # Saturação sem contradição                for r in new:            if frozenset(r) not in set(map(frozenset, [set(c) for c in clauses])):                clauses.append(frozenset(r))print("✅ Motor de inferência por resolução definido.")

#### 4.5 Cenários Sintéticos de Validação

In [5]:
# =====================================================================# CENÁRIOS SINTÉTICOS DE VALIDAÇÃO (TIER_B)# =====================================================================cenarios = [    {        "nome": "Cenário 1: Sem acolhimento",        "desc": "Apenas C=V (contexto comportamental). Sem variáveis nucleares ativas.",        "fatos": [{"C"}],        "esperado": False,    },    {        "nome": "Cenário 2: E ∧ B → A (R3)",        "desc": "Sofrimento emocional + baixo apoio → Acolhimento (regra nuclear).",        "fatos": [{"E"}, {"B"}],        "esperado": True,    },    {        "nome": "Cenário 3: S → A (R1)",        "desc": "Sinal de autoagressão → Acolhimento imediato (protocolo humano).",        "fatos": [{"S"}],        "esperado": True,    },    {        "nome": "Cenário 4: E ∧ B ∧ C → A (R5)",        "desc": "Sofrimento + baixo apoio + contexto comportamental agravante.",        "fatos": [{"E"}, {"B"}, {"C"}],        "esperado": True,    },    {        "nome": "Cenário 5: V ∧ I → A (R6)",        "desc": "Desvalor da vida + insuficiência institucional → Acolhimento.",        "fatos": [{"V"}, {"I"}],        "esperado": True,    },]print("=" * 80)print("VALIDAÇÃO POR CENÁRIOS SINTÉTICOS (TIER_B_SYNTHETIC_SCHOOL)")print("=" * 80)todos_ok = Truefor c in cenarios:    kb = cnf_knowledge_base()    for fato in c["fatos"]:        kb.append(fato)        resultado, steps = resolution_proof(kb)    status = "✅" if resultado == c["esperado"] else "❌"    if resultado != c["esperado"]:        todos_ok = False        print(f"\n{status} {c['nome']}")    print(f"   {c['desc']}")    print(f"   Fatos: {c['fatos']}")    print(f"   A inferido: {resultado} (esperado: {c['esperado']})")    if steps:        print(f"   Passos de resolução:")        for s in steps[-3:]:            print(f"   {s}")print("\n" + "=" * 80)print(f"Resultado: {'✅ TODOS OS CENÁRIOS VALIDADOS' if todos_ok else '❌ FALHAS DETECTADAS'}")print("=" * 80)

#### 4.6 Grafo Determinístico de Explicabilidade

In [6]:
# Gerar e exibir grafo de regrasimport osos.makedirs("../reports/figures", exist_ok=True)kg = build_rule_graph()print(f"Grafo de regras: {kg.entity_count} entidades, {kg.relationship_count} relações")print(f"Variáveis proposicionais: {kg.propositional_variable_count()}")path = export_graph_png(    kg,     "../reports/figures/grafo_regras_7vars.png",    title="Grafo de Regras (7 Variáveis) — AcolheMente Escolar PB")print(f"\n✅ Grafo exportado: {path}")# Exibir inlinefrom IPython.display import Image, displaydisplay(Image(filename=str(path), width=800))

In [7]:
# Gerar grafo de governançakg_gov = build_governance_graph()print(f"Grafo de governança: {kg_gov.entity_count} entidades, {kg_gov.relationship_count} relações")path_gov = export_graph_png(    kg_gov,    "../reports/figures/grafo_governanca_7vars.png",    title="Grafo de Governança de Dados — AcolheMente Escolar PB",    figsize=(16, 10))print(f"\n✅ Grafo exportado: {path_gov}")display(Image(filename=str(path_gov), width=800))

#### 4.7 Governança de Dados: LGPD, ECA, PSE

In [8]:
# Demonstração de governançaprint("=" * 70)print("GOVERNANÇA DE DADOS — AcolheMente Escolar PB")print("=" * 70)guardrails = {    "LGPD": "Nenhum dado identificável processado (Lei 13.709/2018)",    "ECA": "Nenhuma classificação de menor individual (Lei 8.069/1990)",    "PSE": "Integrado como guardrail e encaminhamento (Programa Saúde na Escola)",    "Human-in-the-Loop": "Revisão humana obrigatória antes de qualquer ação",}for g, desc in guardrails.items():    print(f"  ✅ {g}: {desc}")print("\n" + "-" * 70)print("POLÍTICA DE PROVENIÊNCIA (TIERS)")print("-" * 70)tiers = {    "TIER_A_OFFICIAL_BR": "PeNSE 2024 (IBGE) — única fonte inferencial para Brasil/NE/PB",    "TIER_B_SYNTHETIC_SCHOOL": "Dados escolares sintéticos — demonstração operacional",    "TIER_C_EXTERNAL_EXPLORATORY": "Dataset externo — benchmark metodológico APENAS",}for t, desc in tiers.items():    print(f"  📊 {t}: {desc}")print("\n  ❌ MERGE TIER_A × TIER_C: BLOQUEIO ABSOLUTO")print("  ❌ TIER_C não sustenta conclusão sobre Brasil, Nordeste ou Paraíba")

### 5. Resultados e Comparações#### 5.1 Baseline Manual vs. Motor Lógico

In [9]:
# Comparação: baseline manual vs motor lógicoprint("=" * 80)print("COMPARAÇÃO: BASELINE MANUAL vs. MOTOR LÓGICO")print("=" * 80)comparacao = [    ("Critério", "Baseline Manual", "Motor Lógico AcolheMente"),    ("Transparência", "Depende do profissional", "100% auditável (CNF + grafo)"),    ("Reprodutibilidade", "Variável", "Determinístico (mesma entrada → mesma saída)"),    ("Explicabilidade", "Verbal", "Grafo + rastreio de regra"),    ("Linguagem diagnóstica", "Risco de uso", "Bloqueada por guardrails"),    ("Ranking de alunos", "Possível", "Proibido por design"),    ("Governança LGPD", "Não garantida", "Integrada ao motor"),    ("Human-in-the-Loop", "Implícito", "Obrigatório e documentado"),    ("Variáveis", "Ad hoc", "7 formalizadas (5N + 2C)"),    ("Regras", "Tácitas", "6 explícitas em CNF"),]for row in comparacao:    print(f"  {row[0]:<25} │ {row[1]:<30} │ {row[2]}")print("\n" + "=" * 80)

#### 5.2 Rastreabilidade por Regra

In [10]:
# Rastreabilidade: qual regra disparou cada cenárioprint("\nRASTREABILIDADE POR REGRA:")print("-" * 60)rastreio = [    ("S=V", "R1: S → A", "Protocolo humano imediato"),    ("V=V, E=V", "R2: V ∧ E → A", "Desesperança + sofrimento"),    ("E=V, B=V", "R3: E ∧ B → A", "Sofrimento + baixo apoio"),    ("V=V, B=V", "R4: V ∧ B → A", "Desesperança + baixo apoio"),    ("E=V, B=V, C=V", "R5: E ∧ B ∧ C → A", "Sofrimento + baixo apoio + contexto"),    ("V=V, I=V", "R6: V ∧ I → A", "Desesperança + fragilidade institucional"),]for fatos, regra, significado in rastreio:    print(f"  Fatos: {fatos:<18} │ {regra:<22} │ {significado}")

### 6. Perspectiva de Evolução do Trabalho#### 6.1 Evolução teórica com Deep LearningEm uma perspectiva futura, o motor lógico poderia ser complementado por técnicas de deep learning para:- **Detecção de padrões temporais** em dados longitudinais da PeNSE;- **Processamento de linguagem natural** para análise de relatos qualitativos;- **Modelos preditivos** para antecipar cenários de risco.#### 6.2 Governança obrigatória para qualquer evoluçãoQualquer evolução **DEVE** manter:- **LGPD:** Nenhum dado identificável;- **ECA:** Nenhuma classificação de menor;- **PSE:** Integração com rede de saúde;- **Human-in-the-Loop:** Revisão humana obrigatória;- **Sem ranking:** Nenhuma ordenação de estudantes;- **Sem diagnóstico:** Nenhum rótulo clínico automatizado.> **ADR-011:** O motor está congelado em 7 variáveis e 6 regras para esta entrega.> Novas variáveis ou regras só podem ser propostas em versão futura (≥ v0.4.0),> com ADR dedicado e validação ética independente.

### 7. ConclusõesO projeto **AcolheMente Escolar PB** demonstra que é possível construir um sistema de apoio à decisão em saúde mental escolar utilizando **Lógica Proposicional com Inferência por Resolução**, sem recorrer a diagnóstico automatizado, ranking de estudantes ou dependência de APIs externas.**Contribuições principais:**1. Formalização de 7 variáveis proposicionais (5 nucleares + 2 contextuais) a partir da PeNSE 2024;2. Base de conhecimento com 6 regras explícitas em CNF;3. Motor de inferência por resolução 100% determinístico e auditável;4. Grafo de explicabilidade que rastreia cada decisão até as fontes de dados;5. Governança integrada (LGPD, ECA, PSE, Human-in-the-Loop);6. Suite de 72 testes automatizados como guardrails de qualidade;7. Arquitetura de proveniência por tiers com bloqueio absoluto de merge indevido.**Viabilidade:** O sistema é viável para implantação em escolas públicas da Paraíba como ferramenta de suporte, desde que acompanhado de capacitação profissional e integração com a rede PSE/UBS/CAPS.

### 8. Referências Bibliográficas1. RUSSELL, S.; NORVIG, P. *Inteligência Artificial: uma abordagem moderna*. 4ª ed. Rio de Janeiro: GEN LTC, 2022.2. BRASIL. Instituto Brasileiro de Geografia e Estatística (IBGE). **Pesquisa Nacional de Saúde do Escolar (PeNSE) 2024** — Tema 12: Saúde Mental. Rio de Janeiro: IBGE, 2024.3. BRASIL. **Lei nº 13.709, de 14 de agosto de 2018** (Lei Geral de Proteção de Dados Pessoais — LGPD). Brasília: Presidência da República, 2018.4. BRASIL. **Lei nº 8.069, de 13 de julho de 1990** (Estatuto da Criança e do Adolescente — ECA). Brasília: Presidência da República, 1990.5. BRASIL. Ministério da Saúde. **Programa Saúde na Escola (PSE)**. Brasília: MS, 2007.6. BRACHMAN, R. J.; LEVESQUE, H. J. *Knowledge Representation and Reasoning*. Morgan Kaufmann, 2004.7. GENESERETH, M. R.; NILSSON, N. J. *Logical Foundations of Artificial Intelligence*. Morgan Kaufmann, 1987.8. WORLD HEALTH ORGANIZATION. *Mental Health Action Plan 2013–2030*. Geneva: WHO, 2021.